---
title: Parts-Based Decomposition via Disjoint Basis Learning
bibliography: [dictionary-learning.bib, nmf.bib, Pattern_Recognition_and_Machine_Learning.bibtex, gaussian-scaled-mixture.bib]
---

Lately I've been trying my hand at neural network interpretability. This involves looking inside a neural network to understand why it behaves the way it does. I've been stuck at a very specific problem in the current stage of my work.  

> Given a set of vectors, $R$ samples with $C$ dimensions, find a set of basis vectors which can be used to reconstruct the input set as a linear combination, given that for a given dimension $c$, only one basis vector has a non-zero entry, ie. they don't overlap for a given dimensions. 
> For the remaining article, this specific property of no overlap will be called "disjoint support" (if it's the wrong name, please contact me :) )

You can also think of it as a "parts-based representation".   

::: {.callout-warning}
The parts-based representation is not required to have any spatial notion here. Dimensions closer to each other don't cluster in our problem.  
:::

There are many popular algorithms for finding the basis vectors of a given dataset. If your data can be represented linearly, then it can be done using an infinite set of basis vectors. Each algorithm has some constraint to narrow the search. They try to find the basis representations which works best for their usecase. A few examples which were relevant:

- Dictionary learning [@OLSHAUSEN19973311]: The reconstruction is done using a sparse combination of the basis vectors. If your data can be model. The algorithm is widely used in the interpretability field.  
- Non-negative Matrix Factorization [@Lee1999]: The inputs are assumed to be non-negative along with the coefficients, this leads naturally to parts-based representations. Also called NMF.  
- Independent Component Analysis (TODO: CITE): Finding basis vectors which are statistically independent and non-Gaussian. Also called ICA.  

# Some basic notation

I'm not going to work with matrices here. It's easier for me to imagine per sample, atleast for the math. Matrices are useful while coding because of their performance gains. We are given a dataset where:  

- Each sample is a vector $\vec{x_i}$. Each vector has $D$ dimensions, there are $N$ samples in the dataset. We represent it as follows:
- We find a set of $K$ basis vectors, where $\vec{w_k}$ is the $k^{th}$ basis vector. These are common for the whole dataset.  
- We also find a set of coefficient vectors, one for each $\vec{x_i}$, represented by $\vec{s_i}$

$\vec{x_i}$ can be represented using:

$$\vec{x_i} = \sum_{k=0}^{K} s_{ik}\vec{w_k}$$

In probabilistic modeling, I'll simply write this as $\vec{x}=\vec{s} W$ (You can imagine stacking all $w_i$ in a matrix).  


# Test dataset

I first started out by comparing the relevant algorithms on a synthetic test dataset.   

::: {.callout-note}
The dataset generation code creates very clean datasets right now (there is no overlap between disjoint atoms, very less noise). It's not very representative of the real world, but it's a good baseline for now
:::

We create $K$ ground truth vectors, $\vec{w_k}$, along with random $\vec{S}$ sampled from a Gaussian. $\vec{x_i} = \sum_{k=0}^{K} (s_{ik}\vec{w_k}) + \epsilon$. $\epsilon$ is a small random noise, also sampled from Gaussian.  
The ground truth vectors have disjoint support. $\vec{X}$ is generally very clean for now. We create $N$ copies of $\vec{R}$ for generating the dataset $X$

In [ ]:
# | code-fold: true
# | code-summary: Open helper function definitions

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import matplotlib.pyplot as plt
from pt_to_api.utils import show_single_channel_red_green_black as S

MODE = "light"


def make_dim_partition(patch_dim, n_components, seed=42):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(patch_dim)
    return [list(perm[i::n_components]) for i in range(n_components)]


def generate_synthetic_patches(
    patch_dim=72, n_components=10, k=3, n_samples=1000, noise_std=0.01, seed=42
):
    rng = np.random.RandomState(seed)

    dim_partition = make_dim_partition(patch_dim, n_components, seed)

    # ground truth atoms, nonzero only on owned dims
    W_true = np.zeros((n_components, patch_dim))
    for i, dims in enumerate(dim_partition):
        W_true[i, dims] = rng.randn(len(dims))
    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)

    # each sample uses exactly k atoms
    codes_true = np.zeros((n_samples, n_components))
    for i in range(n_samples):
        idx = rng.choice(n_components, k, replace=False)
        codes_true[i, idx] = rng.randn(k)

    X = codes_true @ W_true
    X += rng.randn(*X.shape) * noise_std

    return X, W_true, codes_true, dim_partition


def show_closest_component_of_W_for_each_component(components, W_true, figsize=(5, 2)):
    """
    Given two arrays of numpy vectors of same shapes
    for every component in `components`, this function shows the array in `W_true`
    which has the maximum cosine similarity with the component
    """
    sims = np.abs(cosine_similarity(components, W_true))
    pairs = []
    for i in range(len(components)):
        j = np.argmax(sims[i])
        pairs.append((i, j, sims[i][j]))
    for i, j, score in pairs:
        S(
            [components[i].reshape(3, 3), W_true[j].reshape(3, 3)],
            figsize,
            mode=MODE,
            suptitle=f"similarity score={score}",
            ax_titles=["component", "ground_truth"],
        )
        plt.show()

In [ ]:
# | code-fold: true


X, W_true, codes_true, dim_partition = generate_synthetic_patches(
    patch_dim=9, n_components=3, k=3
)
ws_to_show = [w.reshape(3, 3) for w in W_true]
S(
    ws_to_show,
    (5, 2),
    ncols=3,
    mode=MODE,
    suptitle="ground truth basis vectors \n[bright green = high positive, bright red = high negative, white = near zero]\nA vector is of size 1x9, but is shown as 3x3 for easier visibility",
)
plt.show()
S(
    [X[0].reshape(3, 3)] + ws_to_show,
    (6, 2),
    4,
    suptitle="First input, with its basis components, the coefficient of each component is it's title",
    mode=MODE,
    ax_titles=[
        "X",
        f"{codes_true[0][0]:.3f}",
        f"{codes_true[0][1]:.3f}",
        f"{codes_true[0][2]:.3f}",
    ],
)
plt.show()

# Probabilistic model

Consider the random variables:

- $W$: the collection of basis vectors
- $\vec{s}$: a vector of coefficients for a single input example
- $\vec{x}$: a vector of a single input

We want to find $p(W\vec{s}|\vec{x})$. From Bayesian rule:

$$p(W\vec{s}|\vec{x}) = \frac{p(\vec{x}|W\vec{s})P(W|\vec{s})p(\vec{s})}{p(\vec{x})}$$

$p(\vec{x})$ is a marginalizing constant, and can be ignored.  

$$p(W\vec{s}|\vec{x}) \propto p(\vec{x}|W\vec{s})P(W|\vec{s})p(\vec{s})$$

- We want to model $\vec{x}$ using $W\vec{s} + \epsilon$, where $\epsilon$ is Gaussian noise. 
  - $p(\vec{x}|W\vec{s}) = \mathcal{N}(\vec{x}|W\vec{s}, \sigma_{\epsilon}^2)$.  
- We will assume that $W$ and $\vec{s}$ are sampled independently.  
  - $p(W|\vec{s}) = p(W)$
- I'll also sample $\vec{s}$ from a Gaussian. While optimising, if we don't restrict $\vec{s}$ to some constraints, gradient descent ends up optimising $\vec{s}$ aggressively to create the correct reconstruction. 
  - $p(\vec{s}) = \mathcal{N}(\vec{s}|0, \sigma_s^2)$

$$p(W\vec{s}|\vec{x}) \propto \mathcal{N}(\vec{x}|W\vec{s}, \sigma_{\epsilon}^2) \mathcal{N}(\vec{s}|0, \sigma_s^2) P(W)$$

Everything is pretty standard right now. The important part is constraining $p(W)$ for disjoint support.  

## Constraining $p(W)$ for disjoint support

The goal is simple. There should be no overlap among the basis vectors for a given dimension $c$. We need some way to encode this in $W$'s distribution.  

Until now, we have considered $W$ as the random variable (the set of basis vectors together). Now we want to decide how $W$ is distributed internally. 
$W$ can be thought of as a matrix, with $K$ rows (each row is a basis vector) and $C$ columns (each column is a dimension). An element inside $W$ is written a $w_{rc}$. We want to write the distribution of $W$ in terms of $w_{rc}$. Now at this point, each $w_{rc}$ is also a random variable.  
The generalised product rule gives:

$$p(W) = p(w_{00})p(w_{01}|w_{00})p(w_{02}|w_{00},w_{01}) ... p(w_{RC}|w_{00}w_{01}...w_{R(C-1)})$$

Our problem only requires us to have a dependency when picking weights inside a column (if a column as a non-zero value in some row, the other rows for that column should be 0). There is no dependence among different columns.  
Concretely, let $\vec{w_{c_0}}$ represent the random variable for sampling a vector of numbers for column $0$ (out of $C$ columns) of our matrix $W$.  

::: {.callout-note}
I'm playing a bit around the notation. I honestly can't think of a more simpler notation. This does look confusing though, so I'm just going to talk a bit about the actual intuition.  

Our explicit goal is that we don't want two basis vectors to have non-zero values for a single column.  
Now it makes sense to then think of each column as a probability distribution. Which is why, I added the new notation for describing a random variable which is used to sample numbers for column $i$ as $\vec{w_{c_i}}$. It is a vector because the column consists of a list of numbers. The distribution picks out a vector.  
:::

We can now rewrite $p(W)$ in this form.  
$$p(W) = p(\vec{w_{c_0}})p(\vec{w_{c_1}}|\vec{w_{c_0}})p(\vec{w_{c_2}}|\vec{w_{c_0}}\vec{w_{c_1}})...p(\vec{w_{c_C}}|\vec{w_{c_0}}\vec{w_{c_1}}...\vec{w_{c_{C-1}}})$$

Now, we make the assumption that $\vec{w_{c_i}}$ is independently sampled from $\vec{w_{c_j}}$. This essentially means that column $j$ does not care what values are present in column $i$. We can now simplify:
- $p(\vec{w_{c_j}}|\vec{w_{c_0}}\vec{w_{c_1}}...\vec{w_{c_{j-1}}}) = p(\vec{w_{c_j}})$ for any $j$.  

$$p(W) = \prod_{j=0}^{C} p(\vec{w_{c_j}}) $$



### Constraining each column

The constraint has been delegated to $p(\vec{w_{c_j}})$. I will use $w_{ij}$ to denote a single number at row $i$ for the column $j$. Since we are interested in $p(\vec{w_{c_j}})$, $j$ is now fixed.  

I'm going to do the product rule trick again :).  
$$p(\vec{w_{c_j}}) = p(w_{0j})p(w_{1j}|w_{0j})p(w_{2j}|w_{0j}w_{1j})...p(w_{Rj}|w_{0j}...w_{(R-1)j})$$

My first instinct was something very programmer like:

- $p(w_{1j}|w_{0j})$ would be $\mathcal{N}(0, \sigma_{w_c})$ if $w_{0j}$ is $0$ (we use gaussian sampling if the column until now has no non-zero values).  
- $p(w_{1j}|w_{0j})$ would be $Laplace(0,b)$ if $w_{0j}$ is non-zero. 
  - $Laplace(0,b)$ is famous for introducing sparsity, it maximises the probability of fetching $0$ (in this specific case).  

This can be written maybe in the form of a Bernoulli distribution, a kind of mixture of different distributions. I did try that, along with [Claude](https://claude.ai/).  This can lead to a combinatorial explosion though. It basically creates a tree which splits into two possibilities at each point, which is quite costly.  

Claude suggested using [Gaussian Scale mixtures](https://andrewcharlesjones.github.io/journal/scale-mixtures.html) for this. The original paper is [@10.1214/06-BA117A].  The idea is pretty simple. If you make the $\sigma$ of a Gaussian distribution tiny enough, the probability of sampling the mean shoots up, allowing us to fake sparsity. We can do something like this:

::: {.callout-caution}
Technically, Laplace has the property of giving exact $0$ values when optimising. Replacing it with a Gaussian function with a very low $\sigma$ will give us a high probability of getting values "near $0$", not $0$ itself.  
I don't mind this though, in practice, those near $0$ terms are good enough.  
:::

$$
\begin{align*}
p(w_{1j}|w_{0j}) &= p(w_{1j}|\sigma_{w_{1j}}^2)p(\sigma_{w_{1j}}^2|w_{0j}) \\
&= \mathcal{N}(0, \sigma_{w_{1j}}^2)p(\sigma_{w_{1j}}^2|w_{0j})
\end{align*}
$$

I would like to make this simpler, and simply use some deterministic function $\sigma_{w_{rj}}^2 = f(w_{0j}, w_{1j}...w_{(r-1)j})$. 

- This can be done by assuming $p(\sigma_{w_{rj}}^2|w_{0j},w_{1j}...w_{(r-1)j}) = \delta(\sigma_{w_{rj}}^2 - f(w_{0j}, w_{1j}...w_{(r-1)j}))$
- $\delta(...)$ is the [Dirac delta function](https://en.wikipedia.org/wiki/Dirac_delta_function).  

This gives us

$$
p(w_{rj}|w_{0j},w_{1j}...w_{(r-1)j}) = \mathcal{N}(0, \sigma_{w_{rj}}^2) \quad \text{where} \quad \sigma_{w_{cj}}^2 = f(w_{0j},w_{1j}...w_{(r-1)j})
$$

::: {.callout-note}
I must admit that in the beginning, I had made the substitution naively. It took a while for me to realise that I hadn't followed the basic rules for probability manipulation. I had already finished testing the code, and it was weirdly working.  
I just got lucky that the Dirac Delta function works out. Turns out, it's a standard trick in the literature :)
:::

### Finding $f$

Our basic assumption is we are going to sample $w_{rc}$ from a Gaussian distribution if there are no other $w_{ic} != 0$.  
So in the base case, $p(w_{rc}) = \mathcal{N}(0, \sigma_0^2)$.   

If there is any weight before us which is non-zero, we want $\sigma$ to collapse. So we first want an indicator of whether any of the weight before us is non-zero. A simple solution is to use $\sum_{i=0}^{r-1} |w_{ic}|$ or $\sum_{i=0}^{r-1} w_{ic}^2$. We can use any of them.  

Now if this value is not zero, we want $\sigma$ to collapse to $0$ quickly. There are many functions which provide this behavior. I experimented with:

- Laplace kernel: $\sigma_{x}^2 = \sigma_0^2 e^{\frac{-|x|}{b}}$
- Lorentzian: $\sigma_{x}^2 = \frac{\sigma_0^2}{1 + \alpha x^2}$

These are the plots of these functions for different choice of the hyperparameters.  

In [ ]:
# | code-fold: true
import matplotlib.pyplot as plt

x = np.concatenate([np.linspace(-10, 0, 250), np.linspace(0, 10, 250)])

sigma0_sq = 1.0

b_values = [1, 0.5, 0.1, 0.01]
alpha_values = [1, 10, 100, 1000]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Laplace kernel
ax = axes[0]
for b in b_values:
    y = sigma0_sq * np.exp(-np.abs(x) / b)
    ax.plot(x, y, label=f"b = {b}")
ax.set_title("Laplace Kernel")
ax.set_xlabel("x")
ax.set_ylabel("σ²(x)")
ax.legend()
ax.grid(True, alpha=0.3)

# Lorentzian
ax = axes[1]
for alpha in alpha_values:
    y = sigma0_sq / (1 + alpha * x**2)
    ax.plot(x, y, label=f"α = {alpha}")
ax.set_title("Lorentzian")
ax.set_xlabel("x")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("kernels.png", dpi=150)
plt.show()

I had tried working with Laplace first, but ended up going with Lorentzian as it was more convenient. I've not digged deeper into this.  

Since I chose Lorentzian, I've also chosen to use $x^2 = \sum_{i=0}^{r-1} w_{ic}^2$ in the equation.  

$$
\begin{align*}
\sigma_{w_{rc}}^2 &= \frac{\sigma_{0}^2}{1 + \alpha\sum_{i=0}^{r-1}w_{ic}^2} \\
&= \frac{\sigma_{0}^2}{\phi(w, r-1, c)} \quad \text{where} \quad \phi(w, r, c)=1 + \alpha\sum_{i=0}^{r}w_{ic}^2 \\[14pt]
p(w_{rc}|w_{0c},w_{1c}...w_{r-1,c}) &= \mathcal{N}(0, \sigma_{w_{rc}}^2) \\
&= \frac{1}{\sqrt{2\pi\sigma_{w_{rc}}^2}}\exp\left(\frac{-w_{rc}^2}{2\sigma_{w_{rc}}^2}\right) \\
&= \frac{\phi(w,r-1,c)}{\sigma_0^2}\exp\left(\frac{-w_{rc}^2\phi(w,r-1,c)}{2\sigma_0^2}\right) \\[14pt]
p(\vec{w_{c_j}}) &= p(w_{0j})p(w_{1j}|w_{0j})p(w_{2j}|w_{0j}w_{1j})...p(w_{Rj}|w_{0j}...w_{(R-1)j}) \\[6pt]
&= \prod_{r=0}^{R} \frac{\phi(w,r-1,c)}{\sigma_0^2}\exp\left(\frac{-w_{rc}^2\phi(w,r-1,c)}{2\sigma_0^2}\right) \\
p(W) &= \prod_{j=0}^{C} p(\vec{w_{c_j}}) \\
&= \prod_{j=0}^{C} \prod_{r=0}^{R} \frac{\phi(w,r-1,c)}{\sigma_0^2}\exp\left(\frac{-w_{rc}^2\phi(w,r-1,c)}{2\sigma_0^2}\right)
\end{align*}
$$


::: {.callout-note}
The Gaussian distribution has $\sigma^2$ in the exponent. If you use the Laplace kernel, it looks like this: $e^{\frac{-x^2}{\sigma_0^2e^{\frac{-|w|}{b}}}}$.  
I went with Lorentzian because it "fits" the Gaussian exponent better.  
:::

::: {.callout-important}
$p(\vec{w_{c_j}})$ is a bit "skewed". It prefers the earlier rows to have higher values.  
In practice, at least on the tests I've done, it does not show these properties. I have tested the final loss function by taking an average of the loss derived from this skewed distribution by cycling through all columns as starting points, it gives the same result.  
:::

## The full distribution

$$
\begin{gather*}
p(W\vec{s}|\vec{x}) \propto \mathcal{N}(\vec{x}|W\vec{s}, \sigma_{\epsilon}^2) \mathcal{N}(\vec{s}|0, \sigma_s^2) \prod_{j=0}^{C} \prod_{r=0}^{R}\mathcal{N}(w_{rc}|\sigma_{rc}^2) \\
\sigma_{rc}^2 = \frac{\sigma_0^2}{\phi(w, r-1, c)}
\end{gather*}
$$

# Loss function


The next steps are quite routine. We take the negative log of the probability distribution to get the loss function.  

$$
\begin{align*}
-\ln(p(W)) &= -\ln\left(\prod_{j=0}^{C} \prod_{r=0}^{R}\mathcal{N}(w_{rc}|\sigma_{rc}^2)\right)  \\
&= \frac{1}{2} \sum_{j=0}^{C} \sum_{r=0}^{R} \left[\ln(2\pi) + \ln(\sigma_{w_{rc}}^2) + \frac{w_{rc}^2}{\sigma_{w_{rc}}^2} \right] \\
&= \frac{1}{2} \sum_{j=0}^{C} \sum_{r=0}^{R} \left[ \ln(2\pi) + \ln(\sigma_0^2) - \ln(\phi(w,r-1,c)) + \frac{w_{rc}^2\phi(w,r-1,c)}{\sigma_0^2} \right] \\
-\ln (p(\vec{x}|W\vec{s})) &= \frac{1}{2} \left[\ln(2\pi) + \ln(\sigma_{\epsilon}^2) + \frac{(\vec{x}-W\vec{s})^2}{\sigma_{\epsilon}^2} \right] \\
-\ln (p(\vec{s})) &= \frac{1}{2} \left[ \ln(2\pi) + \ln(\sigma_s^2) + \frac{s^2}{\sigma_s^2} \right] \\
\end{align*}
$$

Removing the terms containing only constants (including the hyperparameters $\sigma_s, \sigma_0, \sigma_{\epsilon}$)

$$
\begin{align*}
-\ln(p(W\vec{s}|\vec{x})) &\propto -\ln(p(\vec{x}|W\vec{s})) - \ln(P(W|\vec{s})) - \ln(p(\vec{s})) \\
&\propto \left[\frac{(\vec{x}-W\vec{s})^2}{\sigma_{\epsilon}^2}\right] + \left[\frac{s^2}{\sigma_s^2} \right] + \left[-\ln(\phi(w,r-1,c)) + \frac{w_{rc}^2\phi(w,r-1,c)}{\sigma_0^2} \right] \\
\end{align*} 
$$

Some simple substitutions, useful later. Most of the loss components are pretty standard.  
$$
\begin{align*}
MSE(X - WS) &=\frac{(\vec{x}-W\vec{s})^2}{\sigma_{\epsilon}^2}  \\
L2(S) &= \frac{s^2}{\sigma_s^2} \\
DisjointLoss(W) &= -\ln(\phi(w,r-1,c)) + \frac{w_{rc}^2\phi(w,r-1,c)}{\sigma_0^2} \\
\end{align*}
$$

## Hyperparameters

We have 4 hyperparameters: $\sigma_{\epsilon}, \sigma_0, \sigma_s, \alpha$.  
These hyperparamaters are quite sensitive, and it is useful to understand what they mean, giving us good starting points.  

$\sigma_0$ and $\alpha$ are very tightly connected, we'll create a simple formula for an $\alpha$ value which works generally.  

### $\sigma_0$ and $\alpha$
Let's first understand the relationship between $\sigma_0$ and $\alpha$.  
$\sigma_0$ is the standard deviation of the Gaussian distribution from which we sample $w_{rc}$, if there is no other non-zero $w_{ic}, i \in [0,c] \setminus{r}$. $\alpha$ is used to shrink this standard deviation for subsequent $w_{ic}, i \in [r+1,c]$


$$\sigma_{w_{rc}}^2 = \frac{\sigma_0^2}{1 + \alpha\sum_{i=0}^{r-1}w_{ic}^2}$$

To understand how $\alpha$ shrinks $\sigma_{w_{rc}}^2$, we can simply analyse:  

$$\sigma_{w_{rc}}^2 = \frac{\sigma_0^2}{1 + \alpha x^2}$$
Basically the lorentzian function, for different $\sigma_0$ and $\alpha$ values.  
The goal is to have $\sigma_{w_{rc}}$ collapse as fast as we can.   

I've plotted two graphs below, first one is for $\sigma_0=1$ and the second one is for $\sigma_0=0.1$. Each of them plot the function using the same set of $\alpha$ values. We notice that for same $\alpha$ values $\sigma_{w_{rc}}$ shrinks slower for $\sigma_0=0.1$.  

### ${\sigma_{\epsilon}}$

The model assumes the error is distributed under $\mathcal{N}(0, \sigma_{\epsilon})$. We want a good estimate of this.  
The easiest way is to simply minimise **only** reconstruction loss ($MSE$). The code uses a very basic autoencoder (covered in the next section). We simply train a baseline autoencoder to do this. And simply use: 
$$\sigma_{\epsilon} = stddev((X - Reconstruction_{baseline}))$$

::: {.callout-caution}
On pure data, without any noise, this can give $0$, which is a bit extreme. I have not explored this case though. We might need to set some empirical minimum value based on the variance of the input dataset
:::

You would want $\sigma_0$ now to be bigger than $\sigma_{\epsilon}$, but still be reasonably small.  
Basic heuristic $\sigma_0 = 5\sigma_{\epsilon}$ works reasonably well empirically.  

### $\sigma_s$

We want $\vec{s}$ to be less constrained than $W$, I've simply set the value to $5\sigma_0$ and it has empirically worked in practice until now. There is no watertight relation though.  

### summary

$$
\begin{align*}
\sigma_{\epsilon} &= stddev((X - Reconstruction_{baseline})) \\
\sigma_0 &= 5\sigma_{\epsilon} \\
\alpha &= \frac{5000}{\sigma_0^2} \\
\sigma_s &= 5\sigma_0 \\
\end{align*}
$$

In [ ]:
# | code-fold: true
# | code-summary: code helper plotting functions

## helper plotting functions
import numpy as np
import matplotlib.pyplot as plt

def _single_plot(ax, sigma_0, alphas, x_limit=None):
    if x_limit is not None:
        lo, hi = x_limit
        x = np.concatenate([np.linspace(lo, 0, 250), np.linspace(0, hi, 251)[1:]])

    for alpha in alphas:
        sigma = np.sqrt(sigma_0**2 / (1 + (alpha * (x**2))))
        ax.plot(x, sigma, label=f"alpha={alpha}")

    ax.set_title(f"sigma_0^2={sigma_0**2:.2e}")
    ax.set_xlabel("x")
    ax.set_ylabel("sigma(x)")
    if x_limit is not None:
        ax.set_xlim(*x_limit)
        ax.set_ylim(0, x_limit[1])
    ax.legend()
    ax.grid(True, alpha=0.3)


def plot_lorentz_for_sigma_0_and_alpha_compare(
    examples
):
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle("plot of sigma_w_rc^2 wrt sigma_0^2 and alpha")
    _single_plot(axes[0], examples[0]["sigma_0"], examples[0]["alphas"], examples[0]["x_limits"])
    _single_plot(axes[1], examples[1]["sigma_0"], examples[1]["alphas"], examples[1]["x_limits"])
    plt.tight_layout()
    plt.show()

In [ ]:
# | code-fold: true

examples = [
    {
        "sigma_0": 1.0,
        "alphas": [0.5, 5.0, 1000, 10_000],
        "x_limits": [-1.0, 1.0],
    },
    {
        "sigma_0": 0.1,
        "alphas": [0.5, 5.0, 1000, 10_000],
        "x_limits": [-0.1, 0.1],
    },
]

plot_lorentz_for_sigma_0_and_alpha_compare(examples)

We need a thinner curve for $\sigma_0=0.01$.  

::: {.callout-important}
$\alpha$ is not a simple scalar. It should be inversely proportional to $x^2$. Since $x$ itself is sampled from $\mathcal{N}(0, \sigma_0^2)$, $x^2$ is in the range $[-3\sigma_0^2, 3\sigma_0^2]$, giving us using $\alpha = \frac{constant}{\sigma_0^2}$ a good heuristic.  
A useful empirical value is the range $constant \in [1000, 10,000]$.  
What this basically does is keep $\alpha x^2$ predictable and stable across different $\sigma_0$ values.  
:::

The next plots use $\frac{\alpha}{\sigma_0^2}$ as the effective $\alpha$ value. Using $\sigma_0 \in [1, 1e-1, 1e-3, 1e-4]$.   
Notice that with this small change, we get identical plots for all $\sigma_0$ values.  

In [ ]:
# | code-fold: true
def _d(a, s):
    return a / (s*s)

examples = [
    {
        "sigma_0": 1.0,
        "alphas": [_d(a, 1.0) for a in [0.5 , 5.0, 1000, 10_000]],
        "x_limits": [-1.0, 1.0],
    },
    {
        "sigma_0": 0.1,
        "alphas": [_d(a, 0.1) for a in [0.5 , 5.0, 1000, 10_000]],
        "x_limits": [-0.1, 0.1],
    },
]

plot_lorentz_for_sigma_0_and_alpha_compare(examples)


examples = [
    {
        "sigma_0": 1e-3,
        "alphas": [_d(a, 1e-3) for a in [0.5 , 5.0, 1000, 10_000]],
        "x_limits": [-1e-3, 1e-3],
    },
    {
        "sigma_0": 1e-4,
        "alphas": [_d(a, 1e-4) for a in [0.5 , 5.0, 1000, 10_000]],
        "x_limits": [-1e-4, 1e-4],
    },
]

plot_lorentz_for_sigma_0_and_alpha_compare(examples)

# An autoencoder with our loss function

The architecture follows a standard autoencoder design, similar to a Sparse Autoencoder (SAE) — a single-layer encoder-decoder network trained to reconstruct its input through a bottleneck.


Given input $\vec{x}$, the encoder projects it to a latent representation $\vec{s} = \vec{x}E$, and the decoder reconstructs it as $\hat{\vec{x}} = \vec{s}W$. The only departure from a vanilla SAE is the loss function.

To map this to code:

- The intermediate activations from the encoder are $\vec{s}$
- The `nn.Linear` decoder layer is $W$
- The L2 penalty is on the encoder weights $E$, not on $\vec{s}$ directly


```{mermaid}
flowchart LR
    X["X (input)"] --> ENC["nn.Linear\n[encoder]"]
    ENC --> ACT["vector: S \n[intermediate activation]"]
    ACT --> DEC["nn.Linear W\n[decoder]"]
    DEC --> RECON["X (reconstruction)"]

    L2["L2(S)"] -. regularizer .-> ENC
    DS["DisjointLoss(W)"] -. regularizer .-> DEC
    MSE["MSE(X-WS)"] -. objective .-> RECON
```


::: {.callout-note}
A more classical solution like how dictionary learning is implemented in `sklearn` with coordinate descent might work too, I've not tested it out.  
:::

## Why L2 on encoder

- In practice, $\sigma_0$ must be kept very small — this tightens the constraint on $W$. Without it, gradient descent ignores the penalty on $W$ entirely and just minimises MSE.

- $\vec{s}$ must be free to grow. The coefficients will generally need to be much larger in magnitude than the basis vectors.
  - Directly regularising $\vec{s}$ is counterproductive: the penalty would shrink its magnitude, making reconstruction impossible — you can't satisfy $\vec{x} = W\vec{s}$ when both $W$ and $\vec{s}$ are vanishingly small.
  - The goal isn't small $\vec{s}$, it's *well-behaved* $\vec{s}$. We want to penalise gradient descent for exploiting $\vec{s}$ in degenerate ways, not for letting it scale.
  - Concretely, we want $\vec{s}$ to be uniform — low variance across its components, not low magnitude.

- In the autoencoder setting, $\vec{s} = \vec{x}E$ where $E$ are the encoder weights. Modelling these as Gaussian is natural: the product of a Gaussian matrix with a fixed input is itself Gaussian, so the distributional assumption carries through cleanly.

## Code for the model

This section is going to be code-heavy. I've not collapsed the sections containing the definition of the autoencoder and it's training schedule, those are important parts.   
I'm also going to run the model on the generated synthetic data, which gives similar results to the ones shown in the first section where I compare this with different algorithms.  

In [ ]:
import torch
from torch import nn
from torch import optim


def _get_weights_loss_on_decoder(model, alpha, sigma_0, weights_algo):
    if weights_algo == "cycled":
        weight_loss = model.weights_loss_cycled(alpha, sigma_0, model.decoder.weight)
    else:
        comp1, comp2 = model.weights_loss(alpha, sigma_0, model.decoder.weight)
        weight_loss = (comp1 + comp2).sum()
    return weight_loss


class Autoencoder(nn.Module):
    def __init__(self, input_dim, n_components):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, n_components, bias=True),
        )
        self.decoder = nn.Linear(n_components, input_dim, bias=False)

    def forward(self, x):
        # coefficients is the vector S
        # self.decoder = W
        # x = X
        # recon = Reconstruction
        coefficients = self.encoder(x)
        recon = self.decoder(coefficients)
        return recon, coefficients

    def recon_loss(self, x, recons, sigma_x):
        """Reconstruction loss. MSE"""
        return self.gauss_loss(x, recons) / (sigma_x * sigma_x)

    def coefficients_loss(self, sigma_s):
        """L2 loss on encoder"""
        return self.gauss_loss(self.encoder[0].weight, 0) / (sigma_s * sigma_s)

    def gauss_loss(self, x, mean):
        loss = (x - mean) ** 2
        return torch.sum(loss, 1).mean()

    def weights_loss_cycled(self, alpha, sigma_0, W):
        """
        This is similar to weight_loss function
        We basically run `weight_loss` starting from every dimension c, and then average them out.
        Useful for testing if there is a bias in the main `weights_loss` function
        """
        K = W.shape[1]
        shift_losses = []
        for s in range(K):
            comp1, comp2 = self.weights_loss(alpha, sigma_0, torch.roll(W, -s, dims=1))
            shift_losses.append((comp1 + comp2).sum())
        return torch.mean(torch.stack(shift_losses))

    def weights_loss(self, alpha, sigma_0, W):
        """Vectorized version the Weight loss"""
        W_sq = W**2  # (C, K)
        cumsum = torch.cumsum(W_sq, dim=1)  # (C, K), cumsum[c,k] = sum W[c,0..k]^2
        phi = alpha * torch.roll(cumsum, 1, dims=1) + 1  # (C, K)
        phi[:, 0] = 1  # k=0: phi_weight(W, c, -1, alpha) = alpha*0 + 1
        comp1 = (W_sq * phi) / (sigma_0 * sigma_0)
        comp2 = -torch.log(phi)
        return comp1, comp2


def train(
    X,
    n_components,
    alpha=5000,
    sigma_eps=0.1,
    sigma_s=1,
    sigma_0=1,
    lr=1e-3,
    epochs=2000,
    batch_size=256,
    weights_algo="cycle",
    verbose=True,
):
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: lorentzian sigma shrinker parameter
    sigma_eps: std of noise in the data after modelling the data as a W@S
    sigma_0: std of W, useful to keep very near 0
    sigma_s: sigma for the gaussian distribution for sampling the encoder weights. It indirectly restricts its outputs (the coefficients) to be gaussian
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    model = Autoencoder(input_dim, n_components)

    # svd initialisation helps, but is not very necessary if the hyperparameters are correctly tuned
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    model.decoder.weight.data = torch.tensor(Vt[:n_components].T, dtype=torch.float32)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]

        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)

            # the main loss computation code, most relevant piece
            recon_loss = model.recon_loss(batch, recon, sigma_eps)
            coefficients_loss = model.coefficients_loss(sigma_s)
            weight_loss = _get_weights_loss_on_decoder(
                model, alpha, sigma_0, weights_algo
            )
            loss = recon_loss + weight_loss + coefficients_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if verbose and epoch % 200 == 0:
            print(
                f"epoch {epoch:4d} | recon_loss {recon_loss:.4f} weight_loss {weight_loss.sum():.4f} coefficients_loss {coefficients_loss:.4f}"
            )

    with torch.no_grad():
        recon, codes = model(X_t)
    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),
        recon.numpy(),
    )


def train_baseline(
    X, n_components, lr=1e-3, epochs=2000, batch_size=256, sigma_x=1, verbose=True
):
    """Train the autoencoder with just reconstruction loss, to find an arbitrary linear model which fits the data

    The main training code requires sigma_eps
    the standard deviation of expected gaussian noise
    when the curve is fitted using Y=WX
    We can generally do a simple sweep of hyperparams
    or use simple heuristics
    If the data is linearly "fittable",
    then we get a good starting point
    using this function. 
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    model = Autoencoder(input_dim, n_components)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]
        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)
            recon_loss = model.recon_loss(batch, recon, sigma_x)
            loss = recon_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if verbose and epoch % 200 == 0:
            print(f"finetune epoch {epoch:4d} | recon_loss {recon_loss:.4f}")

    with torch.no_grad():
        recon, codes = model(X_t)

    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )

## Testing on synthetic data

In [ ]:
#| code-fold: true
#| code-summary: Code for Synthetic data generation script
N_COMPS = 3
N_DIM = 9
X, W_true, codes_true, dim_partition = generate_synthetic_patches(
    N_DIM, N_COMPS, seed=10
)


ws_to_show = [w.reshape(3, 3) for w in W_true]
S(
    ws_to_show,
    (5, 2),
    ncols=3,
    mode=MODE,
    suptitle="ground truth basis vectors \nA vector is of size 1x9, but is shown as 3x3 for easier visibility",
)
plt.show()

S(
    [X[0].reshape(3, 3)] + ws_to_show,
    (6, 2),
    4,
    suptitle="First input, with its basis components, the coefficient of each component is it's title",
    mode=MODE,
    ax_titles=[
        "X",
        f"{codes_true[0][0]:.3f}",
        f"{codes_true[0][1]:.3f}",
        f"{codes_true[0][2]:.3f}",
    ],
)
plt.show()

In [ ]:
#| code-fold: true
#| code-summary: Code for model training script

# we first train a baseline model without any of the regularisations on this dataset
# this would help us find the "best fit"
# this gives us sigma_eps (the std dev of the noise) when the data is fitted using WS
model, codes, components, recon = train_baseline(X, 3, epochs=600, verbose=False)

std_eps = (X - recon).std()
# sigma_0 should be tight, empirically these values are working fine
sigma_0 = std_eps * 5
sigma_s = sigma_0 * 10

# alpha and sigma_0 are tied together. more on this relationship later
alpha = 5000 * (1 / (sigma_0 * sigma_0))
print(
    f"using parameters: sigma_0={sigma_0:.4f} sigma_s={sigma_s:.4f} sigma_eps={std_eps:.4f} alpha={alpha:.4f}"
)

model, codes, components, recon = train(
    X,
    3,
    alpha=alpha,
    sigma_eps=std_eps,
    sigma_0=sigma_0,
    sigma_s=sigma_s,
    weights_algo="no-cycle",
    verbose=False,
)
show_closest_component_of_W_for_each_component(components, W_true)

Lets look at some of the reconstructions and their components

In [ ]:
# | code-fold: true
ws_to_show = [w.reshape(3, 3) for w in components]


i = 0
S(
    [X[i].reshape(3, 3)] + ws_to_show,
    (6, 2),
    4,
    suptitle=f"Reconstruction {i}",
    mode=MODE,
    ax_titles=[
        "X",
        f"{codes_true[i][0]:.3f}",
        f"{codes_true[i][1]:.3f}",
        f"{codes_true[i][2]:.3f}",
    ],
)
plt.show()

i = 1
S(
    [X[i].reshape(3, 3)] + ws_to_show,
    (6, 2),
    4,
    suptitle=f"Reconstruction {i}",
    mode=MODE,
    ax_titles=[
        "X",
        f"{codes_true[i][0]:.3f}",
        f"{codes_true[i][1]:.3f}",
        f"{codes_true[i][2]:.3f}",
    ],
)
plt.show()

i = 1
S(
    [X[i].reshape(3, 3)] + ws_to_show,
    (6, 2),
    4,
    suptitle=f"Reconstruction {i}",
    mode=MODE,
    ax_titles=[
        "X",
        f"{codes_true[i][0]:.3f}",
        f"{codes_true[i][1]:.3f}",
        f"{codes_true[i][2]:.3f}",
    ],
)
plt.show()